In [1]:
import GET_DATA as gd
import pandas as pd
import time

| Categoría                  | Clave       | Ejemplos de valores                                      |
|-----------------------------|-------------|----------------------------------------------------------|
| **Carreteras y transporte** | `highway`   | `motorway`, `primary`, `residential`, `footway`, `cycleway` |
| **Edificios**               | `building`  | `house`, `school`, `hospital`, `church`, `apartments`   |
| **Servicios públicos**      | `amenity`   | `hospital`, `school`, `police`, `fire_station`, `restaurant`, `bank` |
| **Naturaleza**              | `natural`   | `water`, `wood`, `peak`, `beach`, `cliff`, `wetland`    |
| **Uso del suelo**           | `landuse`   | `residential`, `forest`, `industrial`, `farmland`, `commercial` |
| **Turismo y ocio**          | `tourism`   | `hotel`, `museum`, `attraction`, `camp_site`, `zoo`     |
| **Ocio y deporte**          | `leisure`   | `park`, `stadium`, `sports_centre`, `golf_course`, `swimming_pool` |
| **Comercio**                | `shop`      | `supermarket`, `bakery`, `clothes`, `books`, `electronics` |
| **Transporte aéreo y marítimo** | `aeroway`, `waterway` | `runway`, `helipad`, `dock`, `canal`, `ferry_terminal` |
| **Información geográfica**  | `place`     | `city`, `town`, `village`, `hamlet`, `suburb`           |
| **Otros comunes**           | `name`, `addr:*`, `maxspeed`, `oneway`, `access` | nombres, direcciones, restricciones |


In [ ]:

categorias_a_consultar=  {
    'amenity': ['bank', 'hospital', 'school', 'restaurant', 'cafe', 'pharmacy', 'police', 'fuel'],
    'shop': ['supermarket', 'convenience', 'bakery', 'clothes', 'laundry', 'hairdresser'],
    'leisure': ['park', 'playground', 'sports_centre', 'fitness_centre', 'garden'],
    'tourism': ['hotel', 'museum', 'attraction', 'viewpoint', 'art_gallery'],
    'historic': ['monument', 'memorial', 'statue', 'ruins'],
    'highway': ['bus_stop', 'traffic_signals', 'crossing', 'speed_camera'],
    'natural': ['tree', 'water', 'peak'],
    'office': ['government', 'company', 'lawyer', 'estate_agent']
}



area = "Ciudad de México"
fecha = "2025-04-21T00:00:00Z"
lista_df = []

print(f"Iniciando extracción  para: {area}")

for categoria, subcategorias in categorias_a_consultar.items():
    for sub in subcategorias:
        print(f" Consultando: {categoria} = {sub}...", end=" ", flush=True)
        
        # Consulta específica
        df_temp = gd.select_all_from_overpass(
            tries=5,
            timeLimit=60, # Tiempo corto porque la consulta es pequeña
            areaName=area,
            tags={categoria: sub}, 
            date=fecha
        )
        
        if df_temp is not None and not df_temp.empty:
            # Añadimos metadatos para el esquema OLAP
            df_temp['Categoria_Raiz'] = categoria
            df_temp['Clave_Detalle'] = sub
            
            lista_df.append(df_temp)
            print(f"{len(df_temp)} registros.")
        else:
            print("Sin datos.")
        
        # Delay corto para ser un buen ciudadano de internet
        time.sleep(1.5) 

if lista_df:
    # 1. Unir todos los DataFrames. 
    # 'sort=False' evita que Pandas intente reordenar las columnas alfabéticamente al unir.
    df_final = pd.concat(lista_df, ignore_index=True, sort=False)
    
    # 2. Definir el orden de las columnas: Metadatos primero, el resto después.
    meta_cols = ['Categoria_Raiz', 'Clave_Detalle']
    
    # Filtramos solo las columnas que realmente existen en df_final para evitar errores
    exist_meta = [c for c in meta_cols if c in df_final.columns]
    otras_cols = [c for c in df_final.columns if c not in meta_cols]
    
    df_final = df_final[exist_meta + otras_cols]
    
    print("\n" + "="*30)
    print(f"Cubo de información generado con éxito.")
    print(f"Registros totales: {df_final.shape[0]}")
    print(f"Columnas totales: {df_final.shape[1]}")
    print("="*30)
    
    # Guardar el resultado
    gd.save_data(df_final, NombreArchivo=f"Cubo_{area}.csv", CarpetaDestino="C2_ingresosMedios")
else:
    print("\nNo se recolectó ninguna información. Revisa la conexión o las categorías.")

Iniciando extracción  para: Ciudad de México
 Consultando: tourism = hotel... 218 registros.
 Consultando: tourism = museum... 60 registros.
 Consultando: tourism = attraction... 145 registros.
 Consultando: tourism = viewpoint... Intento 1 fallido: servidor ocupado
Intento 2 fallido: servidor ocupado
Intento 3 fallido: servidor ocupado
56 registros.
 Consultando: tourism = art_gallery... Intento 1 fallido: servidor ocupado
Intento 2 fallido: servidor ocupado
Intento 3 fallido: servidor ocupado
Sin datos.
 Consultando: historic = monument... 77 registros.
 Consultando: historic = memorial... Intento 1 fallido: servidor ocupado
